# 🔥 Fire & Smoke Detection System v2 — Deep Learning Training & Grad-CAM Explainable AI Notebook

This notebook trains an **intermediate-level computer vision system** using **MobileNetV2 Transfer Learning**, data augmentation, evaluation metrics (ROC-AUC, Precision, Recall, Confusion Matrix), and **Grad-CAM visual heatmaps**.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")

## 1. Transfer Learning Model Architecture (MobileNetV2)

In [ ]:
def build_fire_smoke_mobilenet(input_shape=(224, 224, 3)):
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False  # Freeze feature extractor

    inputs = tf.keras.Input(shape=input_shape)
    x = preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid', name='fire_prediction')(x)

    model = models.Model(inputs, outputs, name='MobileNetV2_Fire_Smoke')
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    return model

model = build_fire_smoke_mobilenet()
model.summary()

## 2. Grad-CAM Explainable AI Generator

In [ ]:
def compute_gradcam(img_array, model, last_conv_layer_name="Out_ReLU"):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_output, preds = grad_model(img_array)
        class_channel = preds[:, 0]

    grads = tape.gradient(class_channel, last_conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = last_conv_output[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()

print("Grad-CAM Module initialized.")